# Systematic Review
**References:**
* ❌ too old - https://github.com/chandraveshchaudhari/systematic-reviewpy
* ❌ agentic AI - https://github.com/PouriaRouzrokh/LatteReview
* ✅ Uses PubMed API - https://github.com/gijswobben/pymed

**TO-DO**
* ✅ Clean this notebook up
* Run the query on other databases as well
* QC the results from each database, starting with pubmed.
* Once QC'd, merge the results across each database result and process.

**REFERENCES THAT CAUGHT MY EYE**
* https://pubmed.ncbi.nlm.nih.gov/33202965/
* https://pubmed.ncbi.nlm.nih.gov/38389433/

## Environment Setup

In [16]:
# Import all required packages
from datetime import datetime
from dotenv import load_dotenv
from pathlib import Path
from pymed import PubMed
import json
import os
import pandas as pd
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

# Load environment variables
load_dotenv()

True

# Specify Search Strategy

Hack: I used the query builder on https://pubmed.ncbi.nlm.nih.gov/advanced/ to create this




In [ ]:
# This is the bottom-up query that guided the keywords I wanted.
query = '"geospatial analysis" AND "parkinson* disease"' # results =3

In [ ]:
# These are big-to-small queries.
#query = '"(geospatial analysis) AND (parkinson* disease) NOT ((motor) OR (global burden))"' # results = 6
#query = "((geospatial analysis) OR (geographic analysis)) AND (parkinson* disease) AND (environmental atmospheric)"
#query = "(parkinson* disease[title]) AND ((geospatial analysis) OR (pollution)) NOT ((motor) OR (genetic) OR (neurologic) OR (nicotine))" # results = 149
#query = "(parkinson* disease[title]) AND ((geospatial analysis) OR (pollution)) NOT ((motor) OR (genetic) OR (neurologic) OR (nicotine) OR (smoking))" # results = 129

# Query PubMed

In [45]:
# Create a PubMed object that GraphQL can use to query
# Note that the parameters are not required but kindly requested by PubMed Central
# https://www.ncbi.nlm.nih.gov/pmc/tools/developers/
pubmed = PubMed(tool="MyTool", email=os.getenv("PUBMED_EMAIL"))

In [46]:
# Execute the query against the API
# Convert to list so the results can be iterated multiple times (not exhausted)
results = list(pubmed.query(query, max_results=100))

## Initial Query Results

### Preview query results as a DataFrame

This cell converts `results` (a list of PubMed article objects) into a pandas DataFrame for quick inspection. It shows the first 10 rows and creates `df_results` for further exploration. If `results` is missing/empty, the preview cell prints a message instead.

In [47]:
# Preview `results` as a pandas DataFrame
# This cell converts the `results` list (PubMed article objects) into a DataFrame
# and displays a concise preview (first N rows).

if 'results' not in globals() or not results:
    print("`results` is not defined or empty. Run the query cell above to populate `results`.")
else:
    def _article_to_flat_dict(article):
        # Try to leverage article.toJSON() when available
        try:
            raw = article.toJSON()
            if isinstance(raw, str):
                return json.loads(raw)
            if isinstance(raw, dict):
                return raw
        except Exception:
            pass

        # Fallback: extract common attributes safely
        return {
            "pubmed_id": getattr(article, "pubmed_id", None),
            "title": getattr(article, "title", None),
            "publication_date": str(getattr(article, "publication_date", "") or ""),
            "keywords": getattr(article, "keywords", None),
            "abstract": getattr(article, "abstract", None),
        }

    df_results = pd.json_normalize([_article_to_flat_dict(a) for a in results])


In [48]:
# Preview the columns labels as a list before displaying the DataFrame
print(df_results.columns.tolist())

['abstract', 'authors', 'conclusions', 'copyrights', 'doi', 'journal', 'keywords', 'methods', 'publication_date', 'pubmed_id', 'results', 'title', 'xml']


In [51]:
# Pick nice default preview columns (if present)
preview_cols = [c for c in ["title", "abstract", "publication_date", "keywords"] if c in df_results.columns]

# Make output readable in the notebook
pd.set_option('display.max_colwidth', 200)
n = 10
print(f"Showing first {min(n, len(df_results))} of {len(df_results)} results (variable: df_results)")
display(df_results[preview_cols].head(n))

Showing first 3 of 3 results (variable: df_results)


,title,abstract,publication_date,keywords
0,Geospatial Analysis of Persons with Movement Disorders Living in Underserved Regions.,Movement disorders persons from underserved areas have increased barriers to access tertiary care. There is currently limited data on the geographic and demographic profile of movement disorders p...,2021-09-14,"[Movement Disorders, geography, spatial analysis, underserved]"
1,Geospatial analysis of individual-based Parkinson's disease data supports a link with air pollution: A case-control study.,"The etiology of Parkinson's disease (PD) remains unknown. To approach the issue of PD's risk factors from a new perspective, we hypothesized that coupling the geographic distribution of PD with sp...",2021-01-22,"[Air pollution, Environment, Epidemiology, Parkinson's disease, Prevalence, Spatial dependence]"
2,Geospatial Analysis of Environmental Atmospheric Risk Factors in Neurodegenerative Diseases: A Systematic Review.,"Despite the vast evidence on the environmental influence in neurodegenerative diseases, those considering a geospatial approach are scarce. We conducted a systematic review to identify studies con...",2020-11-19,"[environment, epidemiology, geospatial, neurodegenerative, systematic review]"


## Refining the Query
I'll use NLP to identify the keywords to expand the initial query on.

In [52]:
# Combine fields into single text per document
# NOTE: make checks robust against numpy arrays / lists / pandas NA (avoid ambiguous truth values)

def _to_text(row, cols=('title', 'abstract', 'keywords')):
    parts = []
    for c in cols:
        # row might be a dict (to_dict orient='records') or a pandas Series
        if isinstance(row, dict):
            val = row.get(c, '')
        else:
            # Series-like
            val = row.get(c, '') if c in row else ''

        # Handle common NA / empty cases safely (avoid pd.isna() directly in an if)
        if val is None:
            val = ''
        elif isinstance(val, float) and np.isnan(val):
            val = ''
        elif isinstance(val, (list, tuple)):
            # join list-like values
            if len(val) == 0:
                val = ''
            else:
                val = ' '.join(map(str, val))
        elif isinstance(val, (np.ndarray,)):
            # convert numpy arrays to list then join
            if val.size == 0:
                val = ''
            else:
                val = ' '.join(map(str, val.tolist()))
        else:
            # keep whatever string representation
            val = '' if (isinstance(val, float) and np.isnan(val)) else str(val)

        parts.append(val)

    # Only return joined string from non-empty parts
    return ' '.join([p for p in parts if p])

# Small helper to normalize candidate terms when comparing
def _normalize(s):
    s = re.sub(r"[^a-z0-9\s]"," ", s.lower())
    s = re.sub(r"\s+"," ", s).strip()
    return s


In [53]:
# Combine relevant text columns into a single corpus
try:
    corpus = [_to_text(r) for r in df_results.to_dict(orient='records')]
    print(f"Built corpus with {len(corpus)} documents")
    if corpus:
        print("Sample (first document, first 300 chars):\n", corpus[0][:300])
except Exception as e:
    # Provide a helpful debugging hint rather than failing silently
    raise RuntimeError("Failed to build corpus from df_results — check the data types in your text columns") from e

Built corpus with 3 documents
Sample (first document, first 300 chars):
 Geospatial Analysis of Persons with Movement Disorders Living in Underserved Regions. Movement disorders persons from underserved areas have increased barriers to access tertiary care. There is currently limited data on the geographic and demographic profile of movement disorders persons from unders


In [54]:
# Parse out current query words/phrases to avoid suggesting exact duplicates
existing_query_terms = set([_normalize(t) for t in re.findall(r"[A-Za-z*]+(?:\s+[A-Za-z*]+)*", query)])

### TF-IDF and N-gram Frequency Calculation
In information retrieval, tf–idf (term frequency–inverse document frequency, TF*IDF, TFIDF, TF–IDF, or Tf–idf) is a measure of importance of a word to a document in a collection or corpus, adjusted for the fact that some words appear more frequently in general.
https://en.wikipedia.org/wiki/Tf%E2%80%93idf

In [55]:
# TF-IDF over 1..3 grams
tfidf = TfidfVectorizer(ngram_range=(1,3), stop_words='english', max_df=0.9)
X = tfidf.fit_transform(corpus)
feature_names = tfidf.get_feature_names_out()

# Sum TF-IDF across rows to get corpus-level importance
scores = np.asarray(X.sum(axis=0)).ravel()

tfidf_df = pd.DataFrame({'term': feature_names, 'tfidf': scores})
print(f"Computed TF-IDF for {len(tfidf_df)} terms (1-3 grams)")
print(tfidf_df.head(10))

Computed TF-IDF for 1070 terms (1-3 grams)
                    term     tfidf
0                     10  0.034632
1               10 total  0.034632
2           10 total 355  0.034632
3                   1115  0.035829
4          1115 controls  0.035829
5  1115 controls derived  0.035829
6                     12  0.035829
7                 12 614  0.035829
8   12 614 comprehensive  0.035829
9                    121  0.034632


### n-gram
An *n*-gram is a sequence of *n* adjacent symbols in a particular order. The symbols may be *n* adjacent letters, syllables, or rarely whole words found in a language dataset; or adjacent phonemes extracted from a speech-recording dataset, or adjacent base pairs extracted from a genome. They are collected from a text corpus or speech corpus. https://en.wikipedia.org/wiki/N-gram

In [56]:
# Raw frequency counts for same n-grams
cv = CountVectorizer(ngram_range=(1,3), stop_words='english')
Y = cv.fit_transform(corpus)
freqs = np.asarray(Y.sum(axis=0)).ravel()
cv_terms = cv.get_feature_names_out()

freq_df = pd.DataFrame({'term': cv_terms, 'count': freqs})
print(f"Computed raw counts for {len(freq_df)} terms (1-3 grams)")
print(freq_df.head(10))

Computed raw counts for 1075 terms (1-3 grams)
                    term  count
0                     10      1
1               10 total      1
2           10 total 355      1
3                   1115      1
4          1115 controls      1
5  1115 controls derived      1
6                     12      1
7                 12 614      1
8   12 614 comprehensive      1
9                    121      1


In [57]:
# Merge the two recommendation signals
cand = tfidf_df.merge(freq_df, on='term', how='outer').fillna(0)

# Add a combined score that balances TF-IDF and frequency
cand['score'] = cand['tfidf'] * 0.7 + (cand['count'] / (cand['count'].max() + 1e-9)) * 0.3

# Add ngram length (for grouping / filtering)
cand['ngram_len'] = cand['term'].str.count(' ') + 1

# Normalize terms for filtering against the query
cand['term_norm'] = cand['term'].apply(_normalize)

# Filter out terms included in the query and short stopwords
cand = cand[~cand['term_norm'].isin(existing_query_terms)]

# Exclude single characters and pure numbers
cand = cand[cand['term'].str.len() > 2]
cand = cand[~cand['term'].str.match(r"^\d+$")]

In [58]:
# Show top candidates for each n-gram size
top_n = 20
results_unified = cand.sort_values('score', ascending=False).head(top_n)

print(f"Found {len(cand)} candidate terms; showing top {len(results_unified)} overall")

# Display grouped results with helpful columns
display_cols = ['term', 'ngram_len', 'count', 'tfidf', 'score']
print('\nTop candidates (combined score):')
display(results_unified[display_cols].reset_index(drop=True))

# Also show top single-word and multi-word candidates separately
for n in (1,2,3):
    section = cand[cand['ngram_len']==n].sort_values('score', ascending=False).head(12)
    if not section.empty:
        print(f"\nTop {len(section)} ngram_len={n} candidates:")
        display(section[display_cols].reset_index(drop=True))

# Provide a simple function to return a list of recommended terms to extend the query
def suggest_expansions(n=20, min_count=1):
    s = cand[cand['count'] >= min_count].sort_values('score', ascending=False)
    return list(s['term'].head(n))

print('\nExample: call suggest_expansions() to get a list of suggested terms for expanding your query')


Found 1039 candidate terms; showing top 20 overall

Top candidates (combined score):


,term,ngram_len,count,tfidf,score
0,underserved,1,9,0.311687,0.463635
1,neurodegenerative,1,6,0.352000,0.410037
2,spatial,1,7,0.189831,0.323791
3,movement disorders,2,6,0.207791,0.309090
4,movement,1,6,0.207791,0.309090
5,persons,1,6,0.207791,0.309090
6,disorders,1,6,0.207791,0.309090
7,environmental,1,5,0.205719,0.280367
8,prevalence,1,5,0.179144,0.261764
9,population,1,5,0.134423,0.230460



Top 12 ngram_len=1 candidates:


,term,ngram_len,count,tfidf,score
0,underserved,1,9,0.311687,0.463635
1,neurodegenerative,1,6,0.352000,0.410037
2,spatial,1,7,0.189831,0.323791
3,movement,1,6,0.207791,0.309090
4,disorders,1,6,0.207791,0.309090
5,persons,1,6,0.207791,0.309090
6,environmental,1,5,0.205719,0.280367
7,prevalence,1,5,0.179144,0.261764
8,population,1,5,0.134423,0.230460
9,review,1,4,0.160191,0.221225



Top 12 ngram_len=2 candidates:


,term,ngram_len,count,tfidf,score
0,movement disorders,2,6,0.207791,0.309090
1,air pollution,2,4,0.143315,0.209412
2,uf nfind,2,4,0.138527,0.206060
3,environmental atmospheric,2,3,0.176000,0.205018
4,neurodegenerative diseases,2,3,0.176000,0.205018
5,systematic review,2,3,0.176000,0.205018
6,risk factors,2,3,0.116484,0.163357
7,pd prevalence,2,3,0.107486,0.157059
8,spatial dependence,2,3,0.107486,0.157059
9,underserved persons,2,3,0.103896,0.154545



Top 12 ngram_len=3 candidates:


,term,ngram_len,count,tfidf,score
0,atmospheric risk factors,3,2,0.117333,0.136679
1,risk factors neurodegenerative,3,2,0.117333,0.136679
2,factors neurodegenerative diseases,3,2,0.117333,0.136679
3,environmental atmospheric risk,3,2,0.117333,0.136679
4,case control study,3,2,0.071658,0.104706
5,geographic demographic profile,3,2,0.069264,0.103030
6,disorders persons underserved,3,2,0.069264,0.103030
7,persons underserved areas,3,2,0.069264,0.103030
8,movement disorders persons,3,2,0.069264,0.103030
9,analysis 34 included,3,1,0.058667,0.068339



Example: call suggest_expansions() to get a list of suggested terms for expanding your query


## Query with new inclusion criteria

In [ ]:
# This is the expanded query using the NLP-driven term rankings. n = 6,335
# Since risk factors is very general, added an environment term and excluded treatment etc. n = 130
query = '("geospatial analysis" AND "parkinson* disease") OR ("geospatial analysis" AND "movement disorders") OR ("geospatial analysis" AND "neurodegenerative disease*") OR ("parkinson* disease" AND "air pollution") OR ("parkinson* disease" AND "environment*") OR ("parkinson* disease" AND "atmospheric") OR ("parkinson* disease" AND "risk factors" AND "environment*") OR ("parkinson* disease" AND "spatial dependence") NOT ("treatment") NOT ("epidemiology") NOT ("etiology") NOT ("pathology") NOT ("clinical") NOT ("intervention*") NOT ("therap*") NOT ("diagnostic") OR ("parkinson* disease" AND "golf course*")' 

# Execute the query against the API
# Convert to list so the results can be iterated multiple times (not exhausted)
results = list(pubmed.query(query, max_results=200))
df_results = pd.json_normalize([_article_to_flat_dict(a) for a in results])

# Pick nice default preview columns (if present)
preview_cols = [c for c in ["title", "abstract", "publication_date", "keywords"] if c in df_results.columns]

# Make output readable in the notebook
pd.set_option('display.max_colwidth', 200)
n = 10
display(df_results[preview_cols].head(n))

,title,abstract,publication_date,keywords
0,"Low dose ionising radiation elicits MPTP comparable alterations in locomotor Function, oxidative balance and mitochondrial homeostasis in zebrafish embryos.",Prenatal exposure to environmental factors including low-dose ionising radiation and neurotoxins may disrupt the oxidant-antioxidant balance. Our aim was to assess the effects of exposure to low-d...,2025-11-26,"[1-methyl-4-phenyl-1,2,3,6-tetrahydropyridine, Antioxidant, Locomotor activity, Low-dose ionising radiation, Oxidative stress]"
1,Aggregation-prone alpha-synuclein proteoforms and dysregulated molecular signatures in the vermiform appendix of synucleinopathy patients.,"Synucleinopathies, including Parkinson's disease, are neurodegenerative diseases characterized by intracellular inclusions containing the amyloidogenic protein alpha-synuclein. While classically c...",2025-11-24,[]
2,"State-dependent release of extracellular particles with distinct α2,6-sialylation patterns and small RNA cargo related to neuroinflammation.","Neuroinflammation is a significant contributor to neurodegenerative diseases, including Alzheimer's disease, Parkinson's disease, and related dementias; yet peripheral biomarkers for neuroinflamma...",2025-11-24,[]
3,Formation of Condition-Dependent Alpha-Synuclein Fibril Strain in Artificial Cerebrospinal Fluid.,α-Synuclein (aSyn) is an intrinsically disordered protein involved in neurotransmission and synaptic plasticity. The pathological aggregation of this protein is a hallmark of synucleinopathies suc...,2025-11-20,"[Cryo‐EM, Parkinson's disease, aggregate structure analysis, alpha‐synuclein aggregation, physiological conditions]"
4,Do microplastics play a role in the pathogenesis of neurodegenerative diseases? Shared pathophysiological pathways for Alzheimer's and Parkinson's disease.,"The widespread presence of microplastics (MPs) in the environment has raised significant concerns about their potential impact on human health. As of 2023, the Ocean Conservancy estimates that adu...",2025-11-18,"[Alzheimer’s disease, Microplastics, Neuro-pathophysiology, Neurodegenerative diseases, Parkinson’s disease, Plastic environmental pollution]"
5,Elevated blood microplastics and their potential association with Parkinson's disease.,"Microplastic (MP) contamination in human blood and its potential link to Parkinson's disease (PD) remain poorly understood. In this study, we collected whole blood samples from 21 PD patients and ...",2025-11-12,"[Microplastics, Parkinson’s disease, Polyamide 66 (PA66), Polypropylene (PP), Polyvinyl chloride (PVC)]"
6,"Corrigendum to ""Leveraging ANFIS with Adam and PSO optimizers for Parkinson's disease"" [Heliyon Volume 10, Issue 9, May 15, 2024, Article e30241].",[This corrects the article DOI: 10.1016/j.heliyon.2024.e30241.].,2025-11-11,[]
7,Correction: Integrated multi‑omics highlights alterations of gut microbiome functions in prodromal and idiopathic Parkinson's disease.,None,2025-11-08,[]
8,"Heritability and shared environmental effects of brain diseases in 12,040 extended families.","Brain diseases have complex patterns of genetic and environmental risk factors, and better understanding of these risks is required for more effective prevention strategies. Participants of the Du...",2025-11-06,"[Dementia, Genetics, Neurodegenerative diseases, Parkinson's disease, Stroke]"
9,Ferroptosis Induced by Pb Exposure Causes Parkinson's Disease-Related Dopaminergic Neuronal Death.,Lead (Pb) exposure is strongly associated with neurodegenerative diseases such as Parkinson's disease (PD). A prominent pathological feature of PD is the degeneration of dopaminergic (DA) neurons....,2025-10-29,"[Drosophila melanogaster, Parkinson’s disease, dopaminergic neuron, ferroptosis, lead]"


# Export Query Results

In [76]:
# Config
MAX_RESULTS = 150
OUT_DIR = Path("pubmed_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [77]:
# Ensure `results` is available and not an exhausted iterator
try:
    has_items = hasattr(results, "__len__") and len(results) > 0
except NameError:
    has_items = False

if not has_items:
    print("`results` is empty or undefined — running the query to fetch items")
    # Re-run the query and store as list so it can be reused
    results = list(pubmed.query(query, max_results=MAX_RESULTS))

In [78]:
# Timestamped filename to avoid accidental overwrites
timestamp = datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
OUT = OUT_DIR / f"results-{timestamp}.ndjson"
QUERY_FILE = OUT_DIR / f"query-{timestamp}.txt"

In [79]:
def article_to_dict(article):
    # Prefer the object's own JSON if available
    try:
        raw = article.toJSON()
        if isinstance(raw, str):
            return json.loads(raw)
        if isinstance(raw, dict):
            return raw
    except Exception:
        pass

    # Fallback: extract common fields safely
    return {
        "pubmed_id": getattr(article, "pubmed_id", None),
        "title": getattr(article, "title", None),
        "keywords": [k for k in (getattr(article, "keywords", []) or []) if k],
        "publication_date": str(getattr(article, "publication_date", "") or ""),
        "abstract": getattr(article, "abstract", None),
    }

count = 0
with OUT.open("w", encoding="utf-8") as fh:
    for a in results:
        try:
            obj = article_to_dict(a)
            fh.write(json.dumps(obj, ensure_ascii=False))
            fh.write("\n")
            count += 1
        except Exception as e:
            # log and continue
            print(f"Failed to write article {getattr(a, 'pubmed_id', '<unknown>')}: {e}")

In [80]:
# Write the query metadata and query string to a timestamped text file
try:
    with QUERY_FILE.open("w", encoding="utf-8") as qf:
        qf.write(f"timestamp: {timestamp}\n")
        qf.write(f"max_results: {MAX_RESULTS}\n")
        qf.write("query:\n")
        qf.write(query)
except Exception as e:
    print(f"Failed to write query file: {e}")

print(f"Wrote {count} records to {OUT.resolve()}")
print(f"Wrote query file to {QUERY_FILE.resolve()}")

Wrote 200 records to /mnt/c/Users/ReginaChua/Desktop/sysrev/pubmed_results/results-20251201T015259Z.ndjson
Wrote query file to /mnt/c/Users/ReginaChua/Desktop/sysrev/pubmed_results/query-20251201T015259Z.txt


# Get a list of articles
Convert the latest NDJSON to a GitHub-friendly CSV (saved in project root)

In [13]:
# Configure paths - NDJSON in pubmed_results/, CSV in project root
OUT_DIR = Path('pubmed_results')
csv_name = Path(f'results-{datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")}.csv')

In [14]:
# Re-use the DataFrame if it exists, otherwise load from NDJSON
if 'ndjson_df' not in globals():
    # Find latest NDJSON
    files = sorted(OUT_DIR.glob('results*.ndjson'), key=lambda p: p.stat().st_mtime, reverse=True)
    if not files:
        raise FileNotFoundError("No NDJSON files found. Run the export cell first.")
    
    latest = files[0]
    print(f"Loading from {latest.name}")
    
    # Load NDJSON
    records = []
    with latest.open('r', encoding='utf-8') as fh:
        for line in fh:
            if line.strip():
                records.append(json.loads(line))
    
    # Create DataFrame
    ndjson_df = pd.json_normalize(records)
    
    # Convert keywords to strings if present
    if 'keywords' in ndjson_df.columns:
        ndjson_df['keywords'] = ndjson_df['keywords'].apply(lambda k: ', '.join(k) if isinstance(k, (list, tuple)) else k)


In [15]:
# Save as CSV with minimal processing for GitHub readability
try:
    # Reorder columns for readability (put common fields first)
    preferred = ['pubmed_id', 'title', 'publication_date', 'keywords', 'abstract']
    cols = [c for c in preferred if c in ndjson_df.columns] + [c for c in ndjson_df.columns if c not in preferred]
    
    # Write CSV (UTF-8 encoding, no index) to project root
    ndjson_df[cols].to_csv(csv_name, index=False, encoding='utf-8')
    print(f"Saved CSV to project root: {csv_name}")
    
except Exception as e:
    print(f"Error saving CSV: {e}")

Saved CSV to project root: results-20251030T083636Z.csv
